In [1]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

Ultralytics version: 8.3.232
PyTorch version: 2.3.1
GPU: NVIDIA GeForce GTX 1650 SUPER


In [2]:
model = YOLO('yolo11s-seg.pt') 

results = model.train(
    data=r"D:\projeto_placentas_clayton\data.yaml",
    epochs=120,
    imgsz=640,
    batch=2,             # VRAM 4GB
    patience=30,
    device=0,
    name='placentas_v11_aug',
    
    # multitask: contar e medir
    retina_masks=True,   # medicao de bordas hi-res
    overlap_mask=False,  # contagem sem sobreposicao
    mask_ratio=1,        # mantem resolucao da mascara
    
    # augmentation
    degrees=90.0,        
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,          
    mixup=0.1,           
    scale=0.5,           
    hsv_s=0.7,
)

New https://pypi.org/project/ultralytics/8.4.9 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\projeto_placentas_clayton\data.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.1, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=placentas_v11_

In [ ]:
# val9 - extrair metricas programaticamente
import numpy as np
from ultralytics import YOLO

model = YOLO(r'D:\projeto_placentas_clayton\dev\projeto-placentas\runs\segment\placentas_v11_aug\weights\best.pt')

stats = model.val(plots=True, verbose=False)

try:
    x_thresholds = stats.seg.px
    
    y_f1_scores = stats.seg.f1_curve.mean(0)
    
    best_idx = np.argmax(y_f1_scores)
    
    optimal_conf = x_thresholds[best_idx]
    peak_f1_score = y_f1_scores[best_idx]

    print("\n" + "="*50)
    print("OFFICIAL PROGRAMMATIC METRICS EXTRACTION")
    print("="*50)
    print(f"Metrics Source:     SegmentMetrics (Mask)")
    print(f"Peak F1 Score:      {peak_f1_score:.4f}")
    print(f"Optimal Confidence: {optimal_conf:.3f}")
    print("="*50)
    
    box_conf = stats.box.px[np.argmax(stats.box.f1_curve.mean(0))]
    print(f"Reference Box Conf: {box_conf:.3f}")

except Exception as e:
    print(f"Error accessing attributes: {e}")
    x = stats.seg.curves_results[3][1]
    y = stats.seg.curves_results[3][2].mean(0)
    optimal_conf = x[np.argmax(y)]
    print(f"Fallback Conf: {optimal_conf:.3f}")

Ultralytics 8.3.232  Python-3.10.19 torch-2.3.1 CUDA:0 (NVIDIA GeForce GTX 1650 SUPER, 4096MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,203 parameters, 0 gradients
val: Fast image access  (ping: 0.10.0 ms, read: 375.892.1 MB/s, size: 47.6 KB)
val: Scanning D:\projeto_placentas_clayton\dataset_ready_for_yolo\valid\labels.cache... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 6.1s/it 6.1s
                   all         10        327      0.873      0.856      0.893      0.598      0.863      0.847      0.878      0.491
Speed: 5.2ms preprocess, 45.3ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to D:\projeto_placentas_clayton\dev\projeto-placentas\runs\segment\val9

OFFICIAL PROGRAMMATIC METRICS EXTRACTION
Metrics Source:     SegmentMetrics (Mask)
Peak F1 Score:      0.8584
Optimal Conf

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

MODEL_PATH = r'D:\projeto_placentas_clayton\dev\projeto-placentas\runs\segment\placentas_v11_aug\weights\best.pt'
IMAGES_DIR = r"D:\projeto_placentas_clayton\dataset_ready_for_yolo\valid\images"
LABELS_DIR = r"D:\projeto_placentas_clayton\dataset_ready_for_yolo\valid\labels"
CONF = 0.399
AREA_FACTOR = (50 / 72.5) ** 2  # ~0.4757 um2/px

model = YOLO(MODEL_PATH)
comparison_data = []

print(f"Starting Validation Comparison (Conf: {CONF})...")

results = model.predict(source=IMAGES_DIR, conf=CONF, retina_masks=True, verbose=False)

for r in results:
    img_name = os.path.basename(r.path)
    label_path = os.path.join(LABELS_DIR, img_name.replace(os.path.splitext(img_name)[1], '.txt'))
    
    # predicoes modelo (AI)
    ai_count = len(r.boxes)
    ai_area_um2 = (r.masks.data.sum().item() * AREA_FACTOR) if r.masks is not None else 0

    # labels (ground truth - GT)
    gt_count = 0
    gt_area_px = 0
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            gt_count = len(lines)
            
            # calcular area GT
            mask_gt = np.zeros((640, 640), dtype=np.uint8)
            for line in lines:
                parts = list(map(float, line.strip().split()))
                if len(parts) > 1:
                    poly = np.array(parts[1:]).reshape(-1, 2)
                    poly[:, 0] *= 640 # largura
                    poly[:, 1] *= 640 # altura
                    cv2.fillPoly(mask_gt, [poly.astype(np.int32)], 1)
            gt_area_px = np.sum(mask_gt)
    
    gt_area_um2 = gt_area_px * AREA_FACTOR

    # comparacoes
    count_diff = ai_count - gt_count
    area_diff_pct = ((ai_area_um2 - gt_area_um2) / gt_area_um2 * 100) if gt_area_um2 > 0 else 0

    comparison_data.append({
        "Image": img_name,
        "GT_Count": gt_count,
        "AI_Count": ai_count,
        "Count_Err": count_diff,
        "GT_Area_um2": round(gt_area_um2, 1),
        "AI_Area_um2": round(ai_area_um2, 1),
        "Area_Err_%": round(area_diff_pct, 2)
    })

# saida CSV e relatorio
df = pd.DataFrame(comparison_data)
df.to_csv("placenta_disparity_report.csv", index=False)

print("\n" + "="*80)
print(f"{'Image':<20} | {'GT Cnt':<6} | {'AI Cnt':<6} | {'C-Err':<6} | {'Area Error %':<10}")
print("-" * 80)
for _, row in df.iterrows():
    print(f"{row['Image'][:20]:<20} | {int(row['GT_Count']):<6} | {int(row['AI_Count']):<6} | {int(row['Count_Err']):<6} | {row['Area_Err_%']:<10}%")

print("="*80)
print(f"MEAN ABSOLUTE COUNT ERROR: {df['Count_Err'].abs().mean():.2f}")
print(f"MEAN AREA DISPARITY:       {df['Area_Err_%'].mean():.2f}%")
print(f"TOTAL OBJECTS: GT={df['GT_Count'].sum()} | AI={df['AI_Count'].sum()}")
print("="*80)

Starting Validation Comparison (Conf: 0.399)...

Image                | GT Cnt | AI Cnt | C-Err  | Area Error %
--------------------------------------------------------------------------------
ROSILHA-M-B_004_jpg. | 26     | 28     | 2      | 16.8      %
ROSILHA-M-B_008_jpg. | 33     | 27     | -6     | -3.05     %
ROSILHA-M-B_011_jpg. | 17     | 15     | -2     | -0.89     %
TORDILHA-B_006_jpg.r | 26     | 29     | 3      | 12.97     %
TORDILHA-B_007_jpg.r | 39     | 39     | 0      | -0.4      %
TP-B_005_jpg.rf.eee9 | 53     | 52     | -1     | 6.6       %
ZAINA-679-B_001_jpg. | 31     | 30     | -1     | 23.85     %
ZAINA-679-B_006_jpg. | 37     | 33     | -4     | 4.29      %
ZAZA-B_005_jpg.rf.1c | 36     | 32     | -4     | 1.83      %
ZAZA-G_011_jpg.rf.0f | 29     | 31     | 2      | -2.93     %
MEAN ABSOLUTE COUNT ERROR: 2.50
MEAN AREA DISPARITY:       5.91%
TOTAL OBJECTS: GT=327 | AI=316
